## Strategic Patient Risk Stratification & Readmission Predictive Modeling
### Vitality Health Network (VHN) — Clinical Decision Support & Quality Improvement

**Deliverable:** Technical Appendix & Exploratory Data Analysis Notebook (`.ipynb`)  
**Role:** Health Informatics Consulting Group  
**Target Stakeholders:** Chief Medical Information Officer (CMIO), Chief Nursing Officer, Director of Quality & Care Management  

---

#### Executive & Pedagogical Context
Under the ***Hospital Readmissions Reduction Program (HRRP)*** established by the Centers for Medicare & Medicaid Services (CMS), healthcare delivery systems face substantial reimbursement penalties when 30-day all-cause readmission rates for chronic conditions exceed national risk-adjusted benchmarks. 

At ***Vitality Health Network (VHN)***, the historical 30-day readmission rate for diabetic patients has reached **18%**, far outpacing benchmark expectations and incurring millions of dollars in annual penalty exposure. VHN executive leadership has mandated the transition from reactive retrospective reporting to ***proactive, diagnostic, and prescriptive risk stratification***.

This notebook provides the complete analytical pipeline, structured across four rigorous phases:
1. ***Phase 1: Data Ingestion & Clinical Sanitation:*** Ingestion of 100,000+ patient encounters, audit of 50 features, standardizing non-standard missing values (`?`), relational key mapping (`IDs_mapping.csv`), filtering terminal/deceased records, and addressing data quality limitations.
2. ***Phase 2: Data Enrichment via Web Scraping:*** Contextualizing cryptic primary ICD-9 diagnosis codes (`diag_1`) into human-readable clinical disease descriptions using ethical, rate-limited web scraping (`requests` + `BeautifulSoup`).
3. ***Phase 3: Exploratory Data Analysis (EDA):*** Uncovering key clinical, demographic, medication, and operational drivers of readmission; evaluating class imbalance, intersectional disparities, medication regimens (Insulin vs. Oral), and discharge pathways.
4. ***Phase 4: Feature Engineering — The Vitality Complexity Index (VCI):*** Operationalizing an algorithmic bedside risk-scoring tool adapted from the validated LACE Index (Length of stay, Acuity, Comorbidities, Emergency visits) and empirically validating its predictive discrimination.

### Phase 01: Data Ingestion & Clinical Sanitation

#### Project: Patient Risk Stratification – Vitality Health Network (VHN)

#### Phase Overview

This phase focuses on preparing the raw clinical dataset for reliable
data analysis and patient risk stratification.

The main objectives of this phase are:

1. Load the clinical dataset into a Pandas DataFrame.
2. Perform an initial data quality and structure assessment.
3. Standardize non-standard missing values.
4. Analyze missing data.
5. Apply ID mappings to obtain meaningful descriptions.
6. Exclude encounters associated with expired patients.
7. Remove exact duplicate records while preserving valid patient encounters.

#### Expected Outcome

A clean and structured encounter-level dataset that can be
used for the subsequent analytical phases.

In [ ]:
# Import the required libraries
import os
import sys
import io
import time
import re
import warnings
from pathlib import Path
import csv

# Suppress minor deprecation warnings for clean reporting
warnings.filterwarnings('ignore')

# Numerical & Data Manipulation
import numpy as np
import pandas as pd

# Web Scraping Stack
import requests
from bs4 import BeautifulSoup

# Visualization Stack
import matplotlib.pyplot as plt
import seaborn as sns


print("\nRequired libraries imported successfully.")

### 1.1 Loading the Clinical Dataset

The `diabetic_data.csv` file contains patient encounter information
collected from the hospital's electronic health record system.

The dataset is loaded into a Pandas DataFrame for further processing.

The original dataset uses the question mark (`?`) to represent
missing values in certain columns.

The `na_values` parameter converts these non-standard missing-value
markers into standard NumPy `NaN` values during data ingestion.

This ensures consistent missing-value identification throughout
the data cleaning process.

In [ ]:
# Step 1.2: Define the dataset location

data_dir = Path("../data_files")
raw_data_path = data_dir / "diabetic_data.csv"

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {raw_data_path.resolve()}"
    )
  

df_raw = pd.read_csv(
    raw_data_path,
    na_values=["?"],
    low_memory=False
)

print("Clinical dataset loaded successfully!")

### 1.2 Initial Data Assessment

An initial assessment is performed to understand the structure,
features, data types, and statistical characteristics of the dataset.

The following Pandas methods are used:

1. `shape` – Identifies the number of rows and columns.
2. `columns` – Displays all available feature names.
3. `info()` – Provides information about data types and non-null values.
4. `describe()` – Generates descriptive statistics for dataset features.
5. `head()` – Displays the first five records.

Special attention is given to identifier columns because their
numerical values represent categories rather than measurable quantities.

This assessment helps identify potential data quality issues
before applying further cleaning operations.

In [ ]:
# Step 1.2: Perform an initial assessment of the raw dataset

print("Dataset Dimensions:")
print(df_raw.shape)

print("\nDataset Column Names:")
print(df_raw.columns.tolist())

print("\nDataset Information:")
df_raw.info()

print("\nDescriptive Statistical Summary:")
display(df_raw.describe(include="all").T)

print("\nFirst Five Records:")
df_raw.head()

### 1.3 Loading the ID Mapping Dataset

The `IDs_mapping.csv` file contains descriptions for the
categorical identifier columns in the clinical dataset.

These mappings are required to interpret the numerical
identifiers correctly.

The following identifier columns are considered:

1. `admission_type_id`
2. `discharge_disposition_id`
3. `admission_source_id`

The mapping file is structured into separate sections.
Therefore, each section is extracted and stored as a
separate dictionary for further processing.

This approach improves the interpretability of the clinical
dataset and supports the identification of expired patients.

In [ ]:
# Step 1.3: Load and extract ID mappings

mapping_file = data_dir / "IDs_mapping.csv"


if not mapping_file.exists():
    raise FileNotFoundError(
        f"Mapping file not found: {mapping_file.resolve()}"
    )


with mapping_file.open(
    mode="r",
    encoding="utf-8-sig",
    newline=""
) as file:

    mapping_rows = list(csv.reader(file))


required_sections = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]


id_mappings = {}


for section in required_sections:

    mapping_dict = {}
    section_found = False

    for index, row in enumerate(mapping_rows):

        if row and row[0].strip().lower() == section:

            section_found = True

            # Read mapping records until the next section
            for mapping_row in mapping_rows[index + 1:]:

                if not mapping_row or not mapping_row[0].strip():
                    continue

                first_value = mapping_row[0].strip().lower()

                # Stop when another mapping section begins
                if first_value in required_sections:
                    break

                # Store the ID and its description
                if len(mapping_row) >= 2:
                    mapping_id = mapping_row[0].strip()
                    description = mapping_row[1].strip()

                    if mapping_id.isdigit():
                        mapping_dict[mapping_id] = description

            break

    if not section_found:
        raise ValueError(
            f"Mapping section '{section}' was not found."
        )

    id_mappings[section] = mapping_dict


for section, mapping_dict in id_mappings.items():

    print(f"\n{section}")
    print("-" * 50)

    for mapping_id, description in mapping_dict.items():
        print(f"{mapping_id}: {description}")

print("\nID mappings loaded successfully.")

### 1.4 Applying ID Mappings

The numerical identifiers in the clinical dataset represent
categorical information rather than continuous numerical values.

The extracted ID mappings are used to create additional
description columns for better data interpretation.

The following description columns are generated:

- `admission_type_description`
- `discharge_disposition_description`
- `admission_source_description`

The original identifier columns are preserved to maintain
the integrity of the raw clinical data.

These descriptions will also help identify encounters
associated with expired patients.

In [ ]:
# Step 1.4: Apply ID mappings to the clinical dataset


df_cleaned = df_raw.copy()


mapping_columns = {
    "admission_type_id": "admission_type_description",
    "discharge_disposition_id": "discharge_disposition_description",
    "admission_source_id": "admission_source_description"
}


for id_column, description_column in mapping_columns.items():

    id_values = pd.to_numeric(
        df_cleaned[id_column],
        errors="coerce"
    ).astype("Int64").astype("string")


    df_cleaned[description_column] = id_values.map(
        id_mappings[id_column]
    )


for id_column, description_column in mapping_columns.items():

    print(f"\n{id_column} and {description_column}")
    display(df_cleaned[[id_column, description_column]].drop_duplicates().head(15))
    

print("\nID descriptions applied successfully.")

### 1.5 Missing Value Analysis

Missing values can affect the accuracy and reliability
of clinical data analysis.

Although the question mark (`?`) was converted into
standard missing values during data ingestion, a detailed
missing value assessment is required.

This section calculates:

1. Total missing values in each column.
2. Percentage of missing values in each column.
3. Columns containing missing data.

The results are arranged in descending order based on
the percentage of missing values.

This analysis helps identify columns that require
special handling before further data processing.

In [ ]:
# Step 1.5: Analyze missing values in the dataset

# Calculate the total missing values in each column
missing_counts = df_cleaned.isnull().sum()

# Calculate the percentage of missing values in each column
missing_percentages = (
    df_cleaned.isnull().mean() * 100
)


missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages
})

# Sort columns by missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)


print("Missing Value Analysis:")

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
)


print(
    "\nTotal Missing Values:",
    df_cleaned.isnull().sum().sum()
)

### 1.6 Weight Feature Analysis

The `weight` column is examined to determine the extent
of missing information.

A missing value percentage greater than 90% indicates
that the feature contains insufficient information for
reliable analysis.

According to the project requirements, if more than 90%
of the values are missing, the entire `weight` column
must be removed.

This decision is documented as a Data Quality Limitation
because removing the feature results in the loss of
potentially useful clinical information.

If the missing percentage does not exceed 90%, the
column is retained for further consideration.

In [ ]:
# Step 1.6: Analyze and handle missing values in the weight column


if "weight" in df_cleaned.columns:

    weight_missing_percentage = (
        df_cleaned["weight"].isnull().mean() * 100
    )

    print("Weight Column Analysis")
    print("-" * 40)

    print(
        "Total Records:",
        len(df_cleaned)
    )

    print(
        "Missing Values:",
        df_cleaned["weight"].isnull().sum()
    )

    print(
        f"Missing Percentage: "
        f"{weight_missing_percentage:.2f}%"
    )

    # Apply the 90% missing value threshold
    if weight_missing_percentage > 90:

        df_cleaned.drop(
            columns=["weight"],
            inplace=True
        )

        print("\nData Quality Limitation:")
        print(
            "The weight column was removed because "
            "more than 90% of its values were missing."
        )

    else:

        print(
            "\nThe weight column is retained because "
            "its missing percentage does not exceed 90%."
        )

else:

    print(
        "The weight column does not exist in the dataset."
    )

### 1.7 Gender Data Sanitation & Demographic Validation

The `gender` feature represents a critical demographic attribute required for clinical risk stratification, epidemiological profiling, and health equity analysis.

An initial audit of the demographic distribution reveals non-standard entries categorized as `'Unknown/Invalid'`. These values lack clinical validity and demographic utility for downstream predictive modeling.

This step performs the following data sanitation tasks:

1. Inspect the frequency distribution of gender categories before cleaning.
2. Standardize non-valid demographic entries (`'Unknown/Invalid'`) into standard `NaN` values to ensure consistent missing-value tracking.
3. Validate the post-cleaning category distribution and verify the total count of missing values.

This process ensures that invalid entries do not distort demographic summaries, feature encodings, or disparity assessments in subsequent analytical phases.

In [ ]:
# Step 1.7: Standardize invalid entries in the gender column

print("Before cleaning:")
print(df_cleaned['gender'].value_counts(dropna=False))


df_cleaned['gender'] = df_cleaned['gender'].replace('Unknown/Invalid', np.nan)


print("\nAfter cleaning:")
print(df_cleaned['gender'].value_counts(dropna=False))


print("\nMissing gender values:", df_cleaned['gender'].isnull().sum())

### 1.8 Handling Deceased Patients

Patient encounters associated with expired patients must be
excluded before performing readmission analysis.

The `discharge_disposition_id` column contains identifiers
representing the patient's discharge destination or status.

The corresponding descriptions from `IDs_mapping.csv` are
used to identify discharge records associated with expired patients.

All discharge disposition descriptions containing the word
"Expired" are identified.

The corresponding patient encounters are removed from the
working dataset.

The original raw dataset is preserved, and the cleaning
operation is performed on `df_cleaned`.

This step helps prevent deceased patients from being included
in subsequent readmission analysis.

In [ ]:
# Step 1.7: Identify and filter deceased/terminal patients

# Target codes: 11, 19, 20, 21
expired_codes = [11, 19, 20, 21]

deceased_mask = df_cleaned['discharge_disposition_id'].isin(expired_codes)
num_deceased = deceased_mask.sum()
initial_count = len(df_cleaned)

df_cleaned = df_cleaned[~deceased_mask].copy()

records_after_removal = len(df_cleaned) 

print("\nExpired Patient Removal Summary")
print("-" * 45)

print("Records Before Removal:", initial_count)
print("Expired Encounters Removed:", num_deceased)
print("Records After Removal:", records_after_removal)


print("\nExpired patient encounters handled successfully.")

### 1.9 Removing Duplicate Records

Duplicate records can affect the accuracy and reliability
of clinical data analysis.

The dataset is examined to identify exact duplicate rows
using all available columns.

Only records that are completely identical are removed.

Multiple encounters belonging to the same patient are valid
clinical records and must be preserved.

Therefore, deduplication is performed using the entire
record rather than the patient identifier (`patient_nbr`).

The number of duplicate records is calculated before removal,
and the dataset is checked again after the cleaning process.

In [ ]:
# Step 1.9: Identify and remove exact duplicate records

duplicate_count = df_cleaned.duplicated().sum()

print("Duplicate Records Analysis")
print("-" * 40)

print("Records Before Deduplication:", len(df_cleaned))
print("Exact Duplicate Records Found:", duplicate_count)


df_cleaned = df_cleaned.drop_duplicates().copy()

print("\nDeduplication Summary")
print("-" * 40)

print("Records After Deduplication:", len(df_cleaned))

if duplicate_count > 0:
    df_cleaned.drop_duplicates(inplace=True)
    print("Exact duplicates dropped.")

unique_patients = df_cleaned['patient_nbr'].nunique()
print(f"Unique Individual Patients: {unique_patients:,}")

# Distribution of repeat encounters
repeat_counts = df_cleaned['patient_nbr'].value_counts()
print(f"Patients with multiple hospitalizations: {(repeat_counts > 1).sum():,} ({(repeat_counts > 1).sum() / unique_patients * 100:.2f}%)")

print("\nExact duplicate records handled successfully.")

### 1.10 Schema Standardization & Categorical Type Conversion

During initial ingestion, columns containing numeric identifiers (such as `admission_type_id`, `discharge_disposition_id`, and `admission_source_id`) are parsed as integers (`int64`). 

Treating categorical codes as numeric measurements is clinically and statistically invalid:
1. **Mathematical Inappropriateness:** Arithmetic operations (such as computing mean or standard deviation on admission or discharge codes) are meaningless.
2. **Algorithm Distortion:** Machine learning algorithms would misinterpret these discrete categories as continuous numerical scales with intrinsic ordinal rank.
3. **Demographic & Operational Attributes:** Key clinical features (`race`, `gender`, `age`, `payer_code`, `medical_specialty`) represent discrete nominal/ordinal factors that should be formally defined as categories.

In this step, we:
* Enable full column visibility across the Pandas session.
* Convert numerical identifier columns and demographic features into the Pandas `category` data type.
* Verify the updated schema to ensure proper representation and optimized memory utilization.

In [ ]:
# Step 1.10: Convert identifier and demographic columns to categorical data types

print("Show all columns before converted")
print("-" * 50)
print(f"Total columns: {len(df_cleaned.columns)}\n")

with pd.option_context('display.max_rows', None):
    display(df_cleaned.dtypes.to_frame(name='Data Type'))


categorical_columns = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'race',
    'gender',
    'age',
    'payer_code',
    'medical_specialty'
]


for column in categorical_columns:
    df_cleaned[column] = df_cleaned[column].astype('category')


print("Categorical Feature Conversion Summary")
print("-" * 50)
print(f"Total categorical columns converted: {len(categorical_columns)}\n")


display(df_cleaned[categorical_columns].dtypes.to_frame(name="Updated Data Type"))

### 1.11 Identifying the Oral Medication Cohort

According to Section 3.2 and Phase 1 specifications of the project blueprint, the clinical database contains 22 supplementary medication features recorded as individual diabetic drugs (distinct from `insulin`).

These 22 oral agents encompass sulfonylureas, biguanides, thiazolidinediones, meglitinides, alpha-glucosidase inhibitors, and combination therapies:
* **Single Agents:** `metformin`, `repaglinide`, `nateglinide`, `chlorpropamide`, `glimepiride`, `acetohexamide`, `glipizide`, `glyburide`, `tolbutamide`, `pioglitazone`, `rosiglitazone`, `acarbose`, `miglitol`, `troglitazone`, `tolazamide`, `examide`, `citoglipton`.
* **Combination Formulations:** `glyburide-metformin`, `glipizide-metformin`, `glimepiride-pioglitazone`, `metformin-rosiglitazone`, `metformin-pioglitazone`.

In this step, we:
1. Define and verify the presence of all 22 oral medication attributes in `df_cleaned`.
2. Confirm each feature records medication dosage adjustments (`"No"`, `"Steady"`, `"Up"`, `"Down"`).
3. Isolate the list of oral medication columns to support comparative efficacy analysis against Insulin therapy in Phase 3.

In [ ]:
# Step 1.11: Identify, verify, and isolate the 22 oral medication features

# Define the complete list of 22 oral medication columns
oral_medication_columns = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"
]

# Verify column availability in the cleaned dataset
available_oral_meds = [
    col for col in oral_medication_columns 
    if col in df_cleaned.columns
]

missing_oral_meds = [
    col for col in oral_medication_columns 
    if col not in df_cleaned.columns
]

print("Oral Medication Cohort Audit")
print("-" * 50)
print(f"Expected Oral Medications : {len(oral_medication_columns)}")
print(f"Identified in Dataset     : {len(available_oral_meds)}")
print(f"Missing Columns           : {len(missing_oral_meds)}")

if len(available_oral_meds) == 22:
    print("\nVerification Passed: All 22 oral medication features successfully verified.")
else:
    print(f"\nWarning: Missing medication features: {missing_oral_meds}")




print("\nDose Level Distribution for Oral Agents:")
display(df_cleaned[available_oral_meds].apply(pd.Series.value_counts).fillna(0).astype(int))

### 1.12 Final Dataset Assessment

In this step, the cleaned clinical dataset is reviewed to verify the results of the data sanitation process.

The final dataset dimensions, missing values, duplicate records, and data types are examined.

In [ ]:
# Step 1.12: Perform final verification

print("Final Dataset Shape:", df_cleaned.shape)


print("\nRemaining Missing Values:")
print(df_cleaned.isnull().sum().sum())

print("\nRemaining Duplicate Rows:", df_cleaned.duplicated().sum())


print("\nFinal Dataset Information:")
df_cleaned.info()

display(df_cleaned.head())

### Phase 1 Conclusion: Data Ingestion & Clinical Sanitation Summary

Phase 1 established the clinical and technical foundation for the Vitality Health Network (VHN) patient readmission predictive modeling pipeline. Starting with 101,766 raw inpatient encounters, the dataset underwent a structured data hygiene protocol aligned with Centers for Medicare & Medicaid Services (CMS) hospital quality standards.

#### Key Deliverables & Methodological Milestones Achieved:
1. **Missing Data Standardization:** 
   * Non-standard missing indicators (`"?"`) were converted into standard `NaN` representations.
   * The `weight` feature exhibited **96.86% missingness** (98,569 missing encounters). In accordance with the project specification threshold (>90%), `weight` was completely pruned from the feature space and formally cited as a **Data Quality Limitation** to prevent the introduction of severe synthetic imputation bias.
2. **Relational Contextualization:**
   * Parsed relational keys from `IDs_mapping.csv` to map human-readable clinical descriptions for `admission_type_id`, `discharge_disposition_id`, and `admission_source_id`.
3. **Mitigation of Survival Bias:**
   * Identified and filtered **1,652 expired patient encounters** (discharge disposition codes `11`, `19`, `20`, `21`). Deceased patients cannot experience an unplanned 30-day readmission; retaining them would introduce survival bias and artificially deflate predicted readmission rates.
4. **Data Hygiene & Demographic Sanitation:**
   * Audited and removed exact duplicate records while maintaining valid multiple encounters per unique patient (`patient_nbr`).
   * Standardized invalid gender entries (`Unknown/Invalid`) to maintain demographic integrity.
   * Converted categorical identifier codes into Pandas `category` types to prevent mathematical misinterpretation by predictive algorithms.
5. **Therapeutic Cohort Isolation:**
   * Cataloged and verified all **22 oral diabetic medications**, establishing the benchmark cohort for comparative efficacy testing against insulin therapy.

### Phase 2: Data Enrichment via Web Scraping

#### Objective
The `diag_1` column stores each patient's main diagnosis as an ICD-9 code, like `428` or `250.8`. A doctor knows that `428` means heart failure, but a hospital administrator just sees a number. In this phase I fix that by scraping the disease names from a public ICD-9 website and adding them to the dataset, so anyone reading the results can understand them.

#### Why this matters for VHN
- The CMIO asked us to explain *why* readmissions happen, not just report what happened. That is hard to do when the diagnoses are only numbers.
- Readable labels like "Heart failure" make the later phases (EDA, the risk score and the board report) much easier to follow.
- It also shows how an internal dataset can be improved with outside reference data.

#### What I will do in this phase

| Step | Task | Result |
|------|------|--------|
| 2.1 | Find the 20 most common primary diagnosis codes | `top20_codes` list |
| 2.2 | Scrape their descriptions using `requests` and `BeautifulSoup` | ICD-9 lookup table |
| 2.3 | Add the descriptions to the main dataset | `Primary_Diagnosis_Desc` column |

#### Input and output
- ***Input:*** `df_cleaned` from Phase 1 (`?` converted to NaN, `weight` dropped, deceased patients and duplicates removed).
- ***Output:*** the same `df_cleaned` with one new column, `Primary_Diagnosis_Desc`.

### 2.1 Target Identification: Top 20 Primary Diagnoses

#### What I am doing
I count how often each ICD-9 code appears in `diag_1` and select the **20 most frequent** codes. These become the only codes that will be looked up online.

#### Why only the top 20?
1. **Efficiency:** The dataset contains hundreds of distinct `diag_1` codes. Looking up every one would take a long time, because the scraper must wait one second between requests.
2. **Ethical scraping:** Fewer requests means less load on the host server and a lower risk of the IP address being blocked.
3. **Pareto principle:** A small number of conditions usually account for a large share of all hospital encounters. Focusing on them gives the best insight per request.
4. **Project requirement:** The specification explicitly asks for the top 20 codes as the scraping targets.

#### Expected outcome
A ranked table of the 20 codes, and a Python list `top20_codes` which is passed to the scraper in step 2.2.

In [ ]:
# 2.1 Target Identification: Top 20 primary diagnosis codes (diag_1)

TOP_N = 20

top20_df = (df_cleaned['diag_1'].value_counts()
            .head(TOP_N)
            .rename_axis('diag_1')
            .reset_index(name='encounters'))
top20_df.index = top20_df.index + 1 
top20_df.index.name = 'rank'

print(f"Distinct diag_1 codes in the cleaned data: {df_cleaned['diag_1'].nunique():,}")
display(top20_df)

# list that will be passed to the scraper in step 2.2
top20_codes = top20_df['diag_1'].tolist()
print("Scraping targets:", top20_codes)

### Findings from Step 2.1
- The most frequent primary diagnosis is **428** with 6,735 encounters, followed closely by **414** with 6,554. After these two, the counts drop quite a lot: **786** is third with 4,016 encounters.
- The top 20 mixes diabetes codes (`250.8`, `250.6`) with other conditions that often occur alongside diabetes, such as heart-related codes (428, 414, 410, 427) and lung-related codes (486, 491).
- The counts fall steadily down the list. From the 7th code onwards they are fairly close to each other, ranging from about 1,000 to 2,300 encounters.
- Some codes are formatted differently from the others. For example, `38` has lost its leading zero (it should be `038`) and `V57` starts with a letter. I will need to handle this when building the scraper in step 2.2.

**Conclusion:** These 20 codes are my scraping targets for the next step.

### 2.2 Scraper Architecture

#### What I am doing
I build a scraper that looks up the description of each of the 20 target codes on the public ICD-9 website `http://icd9.chrisendres.com/`, using `requests` to download the pages and `BeautifulSoup` to read them.

#### How the scraper works
The website is organised like a book: **chapter → section → code**. For example, code `428` is inside the "Circulatory System" chapter, then inside the "Other Forms of Heart Disease (420-429)" section. The scraper starts at the main index and keeps following the link whose number range contains the code, until it finds the row for that code. For codes with a decimal part (like `250.8`), it goes one level deeper into the page of the main code (`250`).

#### Decisions I made
- **Browsing instead of the search box:** The site's search link did not give reliable results when I tested it, so the scraper follows the index pages instead.
- **Fixing the code format:** Some codes lost their leading zeros in the CSV (for example `38` should be `038`, Septicemia), so I pad numeric codes to 3 digits before looking them up. `V` and `E` codes are left as they are.
- **Ethical scraping:** One second pause (`time.sleep(1)`) after every request, a custom `User-Agent`, a timeout, and a maximum of 3 retries.
- **Error handling:** If one code fails, the scraper reports it and moves on, so one problem does not stop the whole run.
- **Saving results:** The results are saved to a CSV file, so I do not have to scrape again every time I run the notebook.

#### Expected outcome
A lookup table with one description for each of the 20 codes. I will use it in step 2.3.

In [ ]:
# 2.2 Scraper: functions 

import re
from urllib.parse import urljoin

BASE_URL = "http://icd9.chrisendres.com/"
START_URL = BASE_URL + "index.php?action=contents"
HEADERS = {"User-Agent": "Mozilla/5.0 (Academic project - IJSE ITS2122)"}
REQUEST_DELAY = 1     
MAX_RETRIES = 3
TIMEOUT = 10


CODE_ROW = re.compile(r"^([EV]?\d{2,3}(?:\.\d+)?)\s+(.+)$")

RANGE_END = re.compile(r"\(([EV]?\d+)(?:-([EV]?\d+))?\)\s*$")

page_cache = {}      


def normalize_icd9(code):
    """Pad numeric codes to 3 digits: '38' -> '038'. E and V codes stay the same."""
    code = str(code).strip()
    if code[0] in ("E", "V"):
        return code
    whole, _, decimal = code.partition(".")
    whole = whole.zfill(3)
    return f"{whole}.{decimal}" if decimal else whole


def split_code(code):
    """'V57' -> ('V', 57), '428' -> ('', 428)"""
    letter = code[0] if code[0] in ("E", "V") else ""
    return letter, int(float(code[len(letter):]))


def in_range(base, low, high):
    """Is the base code (e.g. '428') inside a section range like 420-429?"""
    b_letter, b_num = split_code(base)
    l_letter, l_num = split_code(low)
    h_letter, h_num = split_code(high)
    return b_letter == l_letter == h_letter and l_num <= b_num <= h_num


def fetch_page(url):
    """Download a page (with retries and a delay). Returns the HTML or None."""
    if url in page_cache:
        return page_cache[url]
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
            response.raise_for_status()
            time.sleep(REQUEST_DELAY)         
            page_cache[url] = response.text
            return response.text
        except requests.exceptions.RequestException as error:
            print(f"   attempt {attempt}/{MAX_RETRIES} failed: {error}")
            time.sleep(REQUEST_DELAY * attempt)
    return None


def parse_rows(html, page_url):
    """Return a list of (first line of text, link or None) for every table row."""
    soup = BeautifulSoup(html, "html.parser")
    rows = []
    for tr in soup.find_all("tr"):
        cells = tr.find_all("td", recursive=False)
        if not cells:
            continue
        cell = cells[-1]                        
        text = cell.get_text("\n", strip=True).replace("\xa0", " ")
        if not text:
            continue
        first_line = text.split("\n")[0].strip()
        link = cell.find("a", href=re.compile("action=child"))
        rows.append((first_line, urljoin(page_url, link["href"]) if link else None))
    return rows


def lookup_icd9(code):
    """Find the description of one ICD-9 code. Returns the text, or None if not found."""
    code = normalize_icd9(code)
    base = code.split(".")[0]
    url, visited = START_URL, set()

    for _ in range(8):                          
        visited.add(url)
        html = fetch_page(url)
        if html is None:
            return None
        rows = parse_rows(html, url)

        for text, link in rows:
            match = CODE_ROW.match(text)
            if match and match.group(1) == code:
                return match.group(2).strip()

        if "." in code and len(code.split(".")[1]) == 1:
            for text, link in rows:
                match = CODE_ROW.match(text)
                if match and match.group(1) == code + "0":
                    return match.group(2).strip()

        next_url = None
        for text, link in rows:
            if not link or link in visited:
                continue
            match = CODE_ROW.match(text)
            if match:
                if code.startswith(match.group(1) + "."):    
                    next_url = link
                    break
            else:
                section = RANGE_END.search(text)
                if section and in_range(base, section.group(1), section.group(2) or section.group(1)):
                    next_url = link
                    break
        if next_url is None:
            return None
        url = next_url
    return None


#  quick test with 1 code before the full run
print("Test 428   ->", lookup_icd9("428"))
print("Test 250.8 ->", lookup_icd9("250.8"))

In [ ]:
# 2.2 Scraper: run on the top 20 codes 

results = []
for i, code in enumerate(top20_codes, start=1):
    print(f"[{i}/{len(top20_codes)}] Looking up {code} ...")
    try:
        description = lookup_icd9(code)
    except Exception as error:             
        print(f"   error for {code}: {error}")
        description = None
    results.append({"diag_1": code, "icd9_description": description})

icd9_lookup = pd.DataFrame(results)

found = icd9_lookup["icd9_description"].notna().sum()
print(f"\nDescriptions found: {found} out of {len(icd9_lookup)}")

failed = icd9_lookup[icd9_lookup["icd9_description"].isna()]["diag_1"].tolist()
if failed:
    print("Codes without a description:", failed)

display(icd9_lookup)

#### Findings from Step 2.2
- The scraper found descriptions for **all 20** codes.
- For example, `428` is **Heart failure** and `414` is **Other forms of chronic ischemic heart disease**.
- The scraper pauses one second after every request, so it takes a minute or two to run.

**Conclusion:** The lookup table is ready to be merged into the main dataset.

### 2.3 Integration: Adding `Primary_Diagnosis_Desc`

#### What I am doing
I add a new column called `Primary_Diagnosis_Desc` to the main dataset. For each encounter, the column shows the plain-English description of its primary diagnosis code (`diag_1`), using the descriptions I scraped in step 2.2.

#### How I label the rows
- If `diag_1` is one of the top 20 codes, it gets the scraped description (for example, `428` becomes "Heart failure").
- If `diag_1` is any other code, or is missing, it gets the label **"Not in Top 20"**, as the project specification asks.
- If a top 20 code was found in the data but the scraper could not find its description, it gets **"Description Unavailable"**. This way a scraping problem is never mixed up with a rare diagnosis.

#### Expected outcome
The same dataset with one extra column. No rows should be added or removed, and the new column should have no empty values.

In [ ]:
# 2.3 Integration: create Primary_Diagnosis_Desc

rows_before = len(df_cleaned)

desc_map = dict(zip(icd9_lookup['diag_1'], icd9_lookup['icd9_description']))


def label_primary_diagnosis(code):
    """Return the description for a top 20 code, otherwise 'Not in Top 20'."""
    if code not in desc_map:
        return "Not in Top 20"
    description = desc_map[code]
    return description if pd.notna(description) else "Description Unavailable"


df_cleaned['Primary_Diagnosis_Desc'] = df_cleaned['diag_1'].apply(label_primary_diagnosis)

assert len(df_cleaned) == rows_before, "Row count changed!"
assert df_cleaned['Primary_Diagnosis_Desc'].isna().sum() == 0, "Empty values found!"

print("Primary_Diagnosis_Desc added successfully.")
print(f"Dataset shape: {df_cleaned.shape}\n")
print(df_cleaned['Primary_Diagnosis_Desc'].value_counts().head(10))

df_cleaned[['encounter_id', 'diag_1', 'Primary_Diagnosis_Desc']].head(10)

#### Findings from Step 2.3
- The new column `Primary_Diagnosis_Desc` was added to the dataset. The number of rows stayed the same (**100,111**) and there are no empty values.
- **"Not in Top 20"** is the largest group with **49,274** encounters (about 49% of the data), because the top 20 codes cover only part of all the diagnoses.
- Among the described diagnoses, the most common is **Heart failure** with **6,735** encounters, followed by **Other forms of chronic ischemic heart disease** (6,554) and **Symptoms involving respiratory system and other chest symptoms** (4,016).

**Conclusion:** The cryptic ICD-9 codes can now be read as disease names, so they can be used in the analysis and the final report.

### Phase 2 Conclusion: Data Enrichment Summary

Phase 2 enriched the dataset by translating cryptic ICD-9 codes in `diag_1` into human-readable clinical descriptions.

#### Summary of Key Achievements:
* **Top 20 Diagnosis Target:** Identified the 20 most frequent admission codes, covering over **50.7% (50,837)** of all encounters.
* **Ethical Web Scraping:** Built a scraper using `requests` and `BeautifulSoup` with polite rate-limiting (1-second delay), error retries, and caching, successfully extracting descriptions for **all 20 codes (100% success rate)**.
* **Feature Integration:** Created the **`Primary_Diagnosis_Desc`** column. All 100,111 encounters were preserved with zero missing values (non-top-20 codes labeled as `"Not in Top 20"`).
* **Key Clinical Insight:** Cardiovascular complications dominate diabetic admissions, led by **Heart failure** (`428` – 6,735 cases) and **Chronic ischemic heart disease** (`414` – 6,554 cases).

**Status:** The enriched dataset is clean, contextualized, and ready for **Phase 3: Exploratory Data Analysis (EDA)**.